This is for to get the coordinates and depth of the detected fish/schools:
1) He/Sp (fish schools)
2) Whiting (individual fish)

Since the data is in 1 second increments, a fish would likely span more than 1 second. This is taken into account. 

Enjoy! :)

In [2]:
import os
import sys
import numpy as np
import pandas as pd

In [3]:
INPUT_PATH = r"C:\Users\tilde\Documents\LSSS\ListUserFile16__F038000_T1_0612.txt"
OUTPUT_PATH = r"C:\Users\tilde\Documents\LSSS"

In [ ]:
COLUMN_NAMES = [
    'YEAR', 'MO', 'DA', 'UTC', 'LOG1', 'LOG2', 'LAT', 'LON',
    'BDMIN', 'BDMAX', 'OBJECT', 'CH', 'PDMIN', 'PDMAX', 'PDMEAN',
    'UPINLM', 'OTHER', 'WHITE', 'OtPl', 'HeSp', 'TOTAL',
]


MAX_GAP_SECONDS = 1

In [5]:
def load_lsss_file(path):
    df = pd.read_csv(path, sep=r'\s+', skiprows=1, names=COLUMN_NAMES)
    df['datetime'] = pd.to_datetime(
        df['YEAR'].astype(str) + '-' +
        df['MO'].astype(str).str.zfill(2) + '-' +
        df['DA'].astype(str).str.zfill(2) + ' ' + df['UTC']
    )
    return df

In [6]:
def group_into_events(hits, label):
    """Group per-second hits (possibly several depth bins per second) of
    one category into events of consecutive seconds."""
    hits = hits.sort_values('datetime').reset_index(drop=True)

    unique_seconds = sorted(hits['datetime'].unique())
    gaps = np.diff(unique_seconds) / np.timedelta64(1, 's')
    new_event = np.concatenate([[True], gaps > MAX_GAP_SECONDS])
    event_id_per_second = np.cumsum(new_event) - 1
    second_to_event = dict(zip(unique_seconds, event_id_per_second))

    hits = hits.copy()
    hits['event_id'] = hits['datetime'].map(second_to_event)

    rows = []
    for _, g in hits.groupby('event_id'):
        sv_col = 'WHITE' if label == 'individual_fish' else 'HeSp'
        sv_max = g[sv_col].max()
        rows.append({
            'category': label,
            'start_time': g['datetime'].min(),
            'end_time': g['datetime'].max(),
            'duration_s': int((g['datetime'].max() - g['datetime'].min())
                               / np.timedelta64(1, 's')) + 1,
            'lat': g['LAT'].mean(),
            'lon': g['LON'].mean(),
            'depth_min_m': g['PDMEAN'].min(),
            'depth_max_m': g['PDMEAN'].max(),
            'sv_sum': g[sv_col].sum(),
            'sv_max': sv_max,
            'sv_max_db': 10 * np.log10(sv_max),
            'n_hits': len(g),
        })
    return rows

In [7]:
def extract_events(df):
    all_rows = []
    for label, col in [('individual_fish', 'WHITE'), ('school', 'HeSp')]:
        hits = df[df[col] > 0]
        if len(hits) == 0:
            continue
        all_rows.extend(group_into_events(hits, label))

    out = pd.DataFrame(all_rows).sort_values('start_time').reset_index(drop=True)
    return out

In [8]:
def main():
    if len(sys.argv) == 3:
        input_path, output_path = sys.argv[1], sys.argv[2]
    else:
        input_path, output_path = INPUT_PATH, OUTPUT_PATH
        if len(sys.argv) != 1:
            print("Usage: python extract_positions.py <input.txt> <output.csv>")
            print("Falling back to INPUT_PATH/OUTPUT_PATH values.")

    if os.path.isdir(output_path):
        output_path = os.path.join(output_path, "fish_positions.csv")

    df = load_lsss_file(input_path)
    events = extract_events(df)
    events.to_csv(output_path, index=False)

    print(f"Saved {len(events)} events to {output_path}")
    print(events['category'].value_counts().to_string())

if __name__ == '__main__':
    main()

Usage: python extract_positions.py <input.txt> <output.csv>
Falling back to INPUT_PATH/OUTPUT_PATH values.
Saved 186 events to C:\Users\tilde\Documents\LSSS\fish_positions.csv
category
individual_fish    138
school              48
